In [1]:
import pandas as pd
from pathlib import Path

pd.set_option('display.max_columns', 120)

path = Path('../data/raw/ontime_2024_01.csv')

In [2]:
preview = pd.read_csv(path, nrows=5)
print(preview.shape)
print(list(preview.columns))

(5, 110)
['Year', 'Quarter', 'Month', 'DayofMonth', 'DayOfWeek', 'FlightDate', 'Reporting_Airline', 'DOT_ID_Reporting_Airline', 'IATA_CODE_Reporting_Airline', 'Tail_Number', 'Flight_Number_Reporting_Airline', 'OriginAirportID', 'OriginAirportSeqID', 'OriginCityMarketID', 'Origin', 'OriginCityName', 'OriginState', 'OriginStateFips', 'OriginStateName', 'OriginWac', 'DestAirportID', 'DestAirportSeqID', 'DestCityMarketID', 'Dest', 'DestCityName', 'DestState', 'DestStateFips', 'DestStateName', 'DestWac', 'CRSDepTime', 'DepTime', 'DepDelay', 'DepDelayMinutes', 'DepDel15', 'DepartureDelayGroups', 'DepTimeBlk', 'TaxiOut', 'WheelsOff', 'WheelsOn', 'TaxiIn', 'CRSArrTime', 'ArrTime', 'ArrDelay', 'ArrDelayMinutes', 'ArrDel15', 'ArrivalDelayGroups', 'ArrTimeBlk', 'Cancelled', 'CancellationCode', 'Diverted', 'CRSElapsedTime', 'ActualElapsedTime', 'AirTime', 'Flights', 'Distance', 'DistanceGroup', 'CarrierDelay', 'WeatherDelay', 'NASDelay', 'SecurityDelay', 'LateAircraftDelay', 'FirstDepTime', 'Total

In [3]:
keep_cols = [
    "Year", "Quarter", "Month", "DayofMonth", "DayOfWeek", "FlightDate",
    "Reporting_Airline", "Tail_Number", "Flight_Number_Reporting_Airline",
    "Origin", "OriginCityName", "OriginState",
    "Dest", "DestCityName", "DestState",
    "CRSDepTime", "DepTime", "DepDelay", "DepDel15", "TaxiOut", "TaxiIn",
    "CRSArrTime", "ArrTime", "ArrDelay", "ArrDel15",
    "Cancelled", "CancellationCode", "Diverted",
    "CRSElapsedTime", "ActualElapsedTime", "AirTime", "Distance",
    "CarrierDelay", "WeatherDelay", "NASDelay", "SecurityDelay", "LateAircraftDelay",
]

missing = [c for c in keep_cols if c not in preview.columns]
print("Not found in file:", missing)

Not found in file: []


In [4]:
df = pd.read_csv(path, usecols=[c for c in keep_cols if c in preview.columns])
print(df.shape)
df.info()

(547271, 37)
<class 'pandas.DataFrame'>
RangeIndex: 547271 entries, 0 to 547270
Data columns (total 37 columns):
 #   Column                           Non-Null Count   Dtype  
---  ------                           --------------   -----  
 0   Year                             547271 non-null  int64  
 1   Quarter                          547271 non-null  int64  
 2   Month                            547271 non-null  int64  
 3   DayofMonth                       547271 non-null  int64  
 4   DayOfWeek                        547271 non-null  int64  
 5   FlightDate                       547271 non-null  str    
 6   Reporting_Airline                547271 non-null  str    
 7   Tail_Number                      541978 non-null  str    
 8   Flight_Number_Reporting_Airline  547271 non-null  int64  
 9   Origin                           547271 non-null  str    
 10  OriginCityName                   547271 non-null  str    
 11  OriginState                      547271 non-null  str    
 12  

In [11]:
nulls = df.isnull().sum().to_frame('null_count')
nulls["null_pct"] = (nulls["null_count"] / len(df) * 100).round(2)
nulls.sort_values('null_pct',ascending=False)
nulls[nulls['null_count'] > 0]

,null_count,null_pct
Tail_Number,5293,0.97
DepTime,19784,3.62
DepDelay,19858,3.63
DepDel15,19858,3.63
TaxiOut,20257,3.70
TaxiIn,20634,3.77
ArrTime,20633,3.77
ArrDelay,21901,4.00
ArrDel15,21901,4.00
CancellationCode,526882,96.27


In [16]:
print(df['Cancelled'].value_counts())
print(df['Diverted'].value_counts())
print(df['CancellationCode'].value_counts(dropna=False))

Cancelled
0.0    526882
1.0     20389
Name: count, dtype: int64
Diverted
0.0    545759
1.0      1512
Name: count, dtype: int64
CancellationCode
NaN    526882
B       12085
A        7736
C         568
Name: count, dtype: int64


In [17]:
key = ["FlightDate", "Reporting_Airline", "Flight_Number_Reporting_Airline",
       "Origin", "Dest", "CRSDepTime"]
print('Duplicate rows on key: ', df.duplicated(subset=key).sum())
print('Date Range: ', df['FlightDate'].min(), 'to', df['FlightDate'].max())
print('Airlines: ', df['Reporting_Airline'].nunique(), '| Airports: ', df['Origin'].nunique())

Duplicate rows on key:  0
Date Range:  2024-01-01 to 2024-01-31
Airlines:  15 | Airports:  334


In [18]:
flown = df[df['Cancelled'] == 0]
print(flown[['DepDelay', 'ArrDelay']].describe())
print('Arrival delay 15+ min: ', round(flown['ArrDel15'].mean() * 100.0, 1), '%')

            DepDelay       ArrDelay
count  526882.000000  525370.000000
mean       15.624411      10.352298
std        63.949686      66.784961
min       -56.000000     -90.000000
25%        -6.000000     -16.000000
50%        -2.000000      -5.000000
75%        12.000000      13.000000
max      3125.000000    3136.000000
Arrival delay 15+ min:  24.1 %


In [22]:
cancelled = df[df['Cancelled'] == 1]
no_code = cancelled['CancellationCode'].isnull().sum()
if no_code > 0:
    warnings.append(f'{no_code} cancelled flights have no cancellation code')